# Bot Question Tracking Spreadsheet
**Date:** 2026-03-07

Extract the "My Score" table from saved FutureEval Bot Tournament HTML page.

- **Input:** `../data/Spring 2026 FutureEval Bot Tournament 03-07-2026.html`
- **Output:** `../products/Bot_Question_Tracking_2026-03-07_v01.csv`

In [1]:
import re
import html
import csv
from pathlib import Path
import pandas as pd

HTML_FILE = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\data\Spring 2026 FutureEval Bot Tournament 03-07-2026.html")
OUTPUT_CSV = Path(r"C:\Users\Donni\projects\metac_bot_Spring_2026\products\Bot_Question_Tracking_2026-03-07_v01.csv")

html_text = HTML_FILE.read_text(encoding="utf-8")
print(f"Loaded {HTML_FILE.name}: {len(html_text):,} chars")

Loaded Spring 2026 FutureEval Bot Tournament 03-07-2026.html: 997,419 chars


In [2]:
# Parse the "My Score" table rows
# Each row: <td><a href=".../questions/NNNNN/">Title</a></td> <th>Coverage</th> <td>Score</td> <th>Weight</th>
row_pattern = re.compile(
    r'href="https://www\.metaculus\.com/questions/(\d+)/">'
    r'(.*?)</a></td>'
    r'<th[^>]*>([^<]*)</th>'
    r'<td[^>]*>([^<]*)</td>'
    r'<th[^>]*>([^<]*)</th>'
)

# Only search in the My Score table area (after "mb-3 w-full")
table_start = html_text.find('class="mb-3 w-full"')
table_end = html_text.find('</table>', table_start)
table_html = html_text[table_start:table_end]

questions = []
for m in row_pattern.finditer(table_html):
    questions.append({
        "question_number": int(m.group(1)),
        "title": html.unescape(m.group(2).strip()),
        "coverage": m.group(3).strip(),
        "score": m.group(4).strip(),
        "question_weight": m.group(5).strip(),
    })

scored = [q for q in questions if q["score"] != "-"]
print(f"Parsed {len(questions)} questions ({len(scored)} scored, {len(questions) - len(scored)} unscored)")
if scored:
    scores = [float(q["score"]) for q in scored]
    print(f"Score range: {min(scores):.3f} to {max(scores):.3f}, total: {sum(scores):.3f}")

Parsed 182 questions (28 scored, 154 unscored)
Score range: -14.535 to 59.838, total: 211.047


In [3]:
# Sort: scored first (descending), then unscored
sorted_qs = sorted(questions, key=lambda q: (q["score"] == "-", -float(q["score"]) if q["score"] != "-" else 0))

# Display as DataFrame
df = pd.DataFrame(sorted_qs)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 80)
df

,question_number,title,coverage,score,question_weight
0,42095,Will any Individual Neutral Athlete (Russian and Belarusian athletes) win an...,100.0%,59.838,1.0
1,41835,"Will the US government enter a shutdown before February 1, 2026?",100.0%,40.187,1.0
2,41846,"Will there be a successful coup in Africa or Latin America before March 1, 2...",100.0%,23.159,1.0
3,41871,What will be NVIDIA's forward guidance in their Q4 FY2026 earnings release? ...,100.0%,21.953,0.7
4,41540,What will Alphabet's Q4 2025 constant-currency revenues be in these global r...,100.0%,19.099,0.7
5,42044,Will Ofcom publish at least one enforcement decision under the UK Online Saf...,100.0%,18.208,1.0
6,41747,What will be the reported Q4 2025 global revenue for Zepbound (tirzepatide)?,100.0%,17.636,1.0
7,42038,What will the Ifo Business Climate Index level (points) for Germany be for F...,100.0%,15.788,1.0
8,41521,Will the UN General Assembly adopt a resolution condemning the US operation ...,100.0%,13.039,1.0
9,41517,"How many dissenting votes will there be at the January 28, 2026 Federal Open...",100.0%,12.216,1.0


In [4]:
# Write CSV
fieldnames = ["question_number", "title", "coverage", "score", "question_weight"]

with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(sorted_qs)

size_kb = OUTPUT_CSV.stat().st_size / 1024
print(f"Wrote {len(sorted_qs)} rows to {OUTPUT_CSV.name} ({size_kb:.1f} KB)")

Wrote 182 rows to Bot_Question_Tracking_2026-03-07_v01.csv (20.6 KB)
